In [ ]:
# Importando bibliotecas
from faker import Faker
from datetime import datetime, timedelta
import re
from PIL import Image
from io import BytesIO
from random import choice, randint, uniform, choices
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from unidecode import unidecode
from hashlib import sha256
from openai import OpenAI

In [ ]:
# Carregando variáveis de ambiente
load_dotenv()

# Acessando o BD principal e o anonimizado, e chave de api da hugging face
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_PRINC')
HUGGING_FACE = os.getenv('HUGGING_FACE')

# Criando engine
entrada = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

engine = create_engine(entrada)

# Carregando faker
f = Faker('pt_BR')

In [ ]:
# Pega o último id das tabelas

def ultimos_ids(conn, tabela):
    result = conn.execute(text(f'SELECT ID FROM {tabela} ORDER BY ID DESC LIMIT 1'))
    return result.scalar()

with engine.connect() as conn:
    tabelas = ['PLANO', 'EMPRESA', 'MIDIA', 'TELEFONE_EMPRESA', 'EMPREGADO', 'SETOR_TURNO', 'MOTIVO', 'EMBALAGEM', 'DESCARTE']
    
    ultimos = {}
    for t in tabelas:
        ultimos[t] = ultimos_ids(conn, t)

In [ ]:
def gerar_observacao(embalagem, motivo):
    prompt = f'Você é um operário que irá registrar uma pequena observação em um descarte de {embalagem} causados por {motivo}, em uma frase sem formatação, se o motivo for \'Outro\', invente. Seja conciso.'
    
    client = OpenAI (
        base_url='https://router.huggingface.co/v1',
        api_key=HUGGING_FACE
    )
    
    completion = client.chat.completions.create (
        model = 'moonshotai/Kimi-K2-Instruct-0905',
        messages = [
            {
                'role': 'user',
                'content': prompt
            }
        ],
        temperature=0.2,
        max_tokens=30
    )
    
    return completion.choices[0].message.content

In [ ]:
# Gerando informações de plano

planos = [
    {"nome": "Standard Mensal", "descricao": "Plano básico, ideal para pequenas indústrias", "preco": 149.99},
    {"nome": "Standard Anual", "descricao": "Plano básico, ideal para pequenas indústrias", "preco": 1439.90},
    {"nome": "Pro Mensal", "descricao": "Plano intermediário, ideal para indústrias em expansão", "preco": 199.99},
    {"nome": "Pro Anual", "descricao": "Plano intermediário, ideal para indústrias em expansão", "preco": 1919.90},
    {"nome": "Premium Mensal", "descricao": "Plano completo, ideal para grandes indústrias", "preco": 399.99},
    {"nome": "Premium Anual", "descricao": "Plano completo, ideal para grandes indústrias", "preco": 3839.90}
]

df_planos = pd.DataFrame(planos)

ultimo_id = (ultimos['PLANO'] or 0) + 1
ids = range(ultimo_id, (ultimo_id + len(df_planos)))
df_planos.insert(0, 'id', ids)

display(df_planos)

In [ ]:
# Gerando informações de empresa

empresas = []

for i in range(20):
    empresa = f.company()
    empresa_email = re.sub('[./-]', ' ', empresa.lower()).strip().replace(' ', '_')
    dominios = ['@email.com.br', '@email.br', '@email.org.br', '@email.com']

    empresas.append({
        "nome": (empresa + ' ' + f.company_suffix()),
        "email": (empresa_email + choice(dominios)),
        "cnpj": (f.cnpj()),
        "uf": (f.estado_sigla()),
        "cidade": (f.city()),
        "bairro": (f.neighborhood()),
        "cep": (f.postcode()),
        "rua": (f.street_name()),
        "numero": (f.building_number()),
        "cod_plano": choice(df_planos['id'])
    })

df_empresas = pd.DataFrame(empresas)

ultimo_id = (ultimos['EMPRESA'] or 0) + 1
ids = range(ultimo_id, (ultimo_id + len(df_empresas)))
df_empresas.insert(0, 'id', ids)

display(df_empresas)

In [ ]:
# Gerando informações de mídia

midias = []

for i in range(20):
    imagem = Image.new('RGB', (800, 600))
    memoria = BytesIO()
    imagem.save(memoria, format="jpeg")
    
    bytes_imagem = memoria.getvalue()
    memoria.close()
    
    midias.append({
        "caminho_arquivo": f.file_path(depth=2, category='image', extension='jpeg'),
        "arquivo": bytes_imagem,
        "horario": f.date_time_this_year()
    })

df_midias = pd.DataFrame(midias)

ultimo_id = (ultimos['MIDIA'] or 0) + 1
ids = range(ultimo_id, (ultimo_id + len(df_midias)))
df_midias.insert(0, 'id', ids)

display(df_midias)

In [ ]:
# Gerando informações de telefone

telefones = []

for i in range (50):
    telefones.append({
        "telefone": (f.cellphone_number()),
        "cod_empresa": choice(df_empresas['id'])
    })

df_telefones = pd.DataFrame(telefones)

ultimo_id = (ultimos['TELEFONE_EMPRESA'] or 0) + 1
ids = range(ultimo_id, (ultimo_id + len(df_telefones)))
df_telefones.insert(0, 'id', ids)

display(df_telefones)

In [ ]:
# Gerando informações de empregado

empregados = []
empregos = ["Operador de Produção", "Operador de Palletização", "Auxiliar de Almoxarifado", "Conferente de Materiais", "Separador de Carga", "Motorista de Empilhadeira", "Técnico de Manutenção", "Analista de Controle de Estoque", "Supervisor de Logística", "Gestor de Embalagens", "Coordenador de Produção"]
probabilidades = [30, 20, 15, 10, 8, 5, 4, 3, 3, 3, 1]

for i in range (20):
    nome = f.first_name()
    sobrenome = " " + f.last_name()
    email = unidecode(nome.lower().replace(' ', '_').replace('.', '') + sobrenome.lower().strip().replace(' ', '_').replace('.', ''))
    cargo = choices(empregos, weights=probabilidades, k=1)[0]
    
    if cargo in ['Analista de Controle de Estoque', 'Supervisor de Logistica', 'Gestor de Embalagens']:
        tipo = 'GESTOR'
    else:
        tipo = 'OPERARIO'
    
    empregados.append({
        "nome": nome,
        "sobrenome": sobrenome,
        "cpf": f.cpf(),
        "email": email + choice(dominios),
        "senha": sha256(f.password().encode()).hexdigest(),
        "telefone_empregado": f.cellphone_number(),
        "cargo": cargo,
        "tipo": tipo,
        "uf": (f.estado_sigla()),
        "cidade": (f.city()),
        "bairro": (f.neighborhood()),
        "cep": (f.postcode()),
        "rua": (f.street_name()),
        "numero": (f.building_number()),
        "cod_empresa": choice(df_empresas['id'])
    })

df_empregados = pd.DataFrame(empregados)

ultimo_id = (ultimos['EMPREGADO'] or 0) + 1
ids = range(ultimo_id,  (ultimo_id + len(df_empregados)))
df_empregados.insert(0, 'id', ids)

display(df_empregados)

In [ ]:
# Gerando setor e turno

setor_turno = []

for i in range(10):
    hora_inicio = f.time()
    while hora_inicio > '15:59:59':
        hora_inicio = f.time()
    
    datetime_inicio = datetime.strptime(hora_inicio, '%H:%M:%S')
    duracao = randint(5, 8)
    datetime_fim = datetime_inicio + timedelta(hours=duracao)
    
    hora_fim = datetime_fim.strftime('%H:%M:%S')
    
    setor_turno.append({
        "nome": f'Setor {i+1}',
        "hora_inicio": hora_inicio,
        "hora_fim": hora_fim
        })

df_setor_turno = pd.DataFrame(setor_turno)

ultimo_id = (ultimos['SETOR_TURNO'] or 0) + 1
ids = range(ultimo_id,  (ultimo_id + len(df_setor_turno)))
df_setor_turno.insert(0, 'id', ids)

display(df_setor_turno)

In [ ]:
# Gerando motivos

motivos = [
    {"nome": "Contaminação", "descricao": "Resíduos de produtos anteriores na embalagem"},
    {"nome": "Falha de Selagem", "descricao": "Vazamento de embalagem a vácuo"},
    {"nome": "Superaquecimento", "descricao": "Geração de calor excessiva pela ciclagem"},
    {"nome": "Desalinhamento", "descricao": "Adesivos/Impressão/Carimbagem errada"},
    {"nome": "Outro", "descricao": "Problema geral na embalagem"}
]

df_motivos = pd.DataFrame(motivos)

ultimo_id = (ultimos['MOTIVO'] or 0) + 1
ids = range(ultimo_id,  (ultimo_id + len(df_motivos)))
df_motivos.insert(0, 'id', ids)

display(df_motivos)

In [ ]:
# Gerando embalagem
embalagens = [
    {"nome": "Garrafa PET", "tipo": "Plástico", "fornecedor": f.company(), "valor_kg": round(uniform(0.0, 10.0), 2)},
    {"nome": "Lata de Refrigerante", "tipo": "Metal", "fornecedor": f.company(), "valor_kg": round(uniform(0.0, 10.0), 2)},
    {"nome": "Caixa de Papelão", "tipo": "Papelão", "fornecedor": f.company(), "valor_kg": round(uniform(0.0, 10.0), 2)},
    {"nome": "Embalagem Plástica", "tipo": "Plástico PEAD", "fornecedor": f.company(), "valor_kg": round(uniform(0.0, 10.0), 2)},
    {"nome": "Embalagem de Papel", "tipo": "Papel Reciclado", "fornecedor": f.company(), "valor_kg": round(uniform(0.0, 10.0), 2)}
]

df_embalagens = pd.DataFrame(embalagens)

ultimo_id = (ultimos['EMBALAGEM'] or 0) + 1
ids = range(ultimo_id,  (ultimo_id + len(df_embalagens)))
df_embalagens.insert(0, 'id', ids)

display(df_embalagens)

In [ ]:
# Gerando informações de descarte

descartes = []

for i in range(10):
    descarte_kg = round(uniform(0.5, 10.0), 1)
    cod_emabalagem = choice(df_embalagens['id'])
    cod_motivo = choice(df_motivos['id'])
    
    valor_kg = df_embalagens.loc[df_embalagens['id'] == cod_emabalagem, 'valor_kg'].values[0]
    motivo = df_motivos.loc[df_motivos['id'] == cod_motivo, 'nome'].values[0]
    embalagem = df_embalagens.loc[df_embalagens['id'] == cod_emabalagem, 'nome'].values[0]
        
    descartes.append({
        "descarte_kg": descarte_kg,
        "custo_estimado": round((descarte_kg * valor_kg), 2),
        "horario": f.date_time_this_year(),
        "observacao": choice([gerar_observacao(embalagem, motivo), None]),
        "cod_embalagem": cod_emabalagem,
        "cod_motivo": cod_motivo,
        "cod_empregado": choice(df_empregados['id']),
        "cod_midia": choice([choice(df_midias['id']), None])
    })

df_descartes = pd.DataFrame(descartes)

ultimo_id = (ultimos['DESCARTE'] or 0) + 1
ids = range(ultimo_id,  (ultimo_id + len(df_descartes)))
df_descartes.insert(0, 'id', ids)

display(df_descartes)

In [ ]:
# Gerando informações de relacionamento

armazena = []
trabalha_em = []

for i in range(15):
    armazena.append ({
        "cod_embalagem": choice(df_embalagens['id']),
        "cod_setor_turno": choice(df_setor_turno['id'])
    })

for i in range (50):
    trabalha_em.append({
        "cod_setor_turno": choice(df_setor_turno['id']),
        "cod_empregado": choice(df_empregados['id'])
    })

df_armazena = pd.DataFrame(armazena)
df_trabalha_em = pd.DataFrame(trabalha_em)

print('ARMAZENA')
display(df_armazena)

print('\nTRABALHA EM')
display(df_trabalha_em)


In [ ]:
# Inserindo no banco de dados
df_planos.to_sql('plano', engine, if_exists='append', index=False)
df_empresas.to_sql('empresa', engine, if_exists='append', index=False)
df_midias.to_sql('midia', engine, if_exists='append', index=False)
df_telefones.to_sql('telefone_empresa', engine, if_exists='append', index=False)
df_empregados.to_sql('empregado', engine, if_exists='append', index=False)
df_setor_turno.to_sql('setor_turno', engine, if_exists='append', index=False)
df_motivos.to_sql('motivo', engine, if_exists='append', index=False)
df_embalagens.to_sql('embalagem', engine, if_exists='append', index=False)
df_descartes.to_sql('descarte', engine, if_exists='append', index=False)
df_armazena.to_sql('armazena', engine, if_exists='append', index=False)
df_trabalha_em.to_sql('trabalha_em', engine, if_exists='append', index=False)